In [ ]:
%pip install keras-tcn openpyxl --quiet

In [ ]:
import os,time,warnings,gc
import numpy as np
import pandas as pd
import glob
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
from scipy.ndimage import gaussian_filter1d
from scipy.interpolate import interp1d
from scipy.signal import find_peaks
from scipy.stats import skew,pearsonr
from sklearn.model_selection import train_test_split
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers,Model
from tensorflow.keras.callbacks import EarlyStopping,ModelCheckpoint
from tcn import TCN
warnings.filterwarnings('ignore')
tf.random.set_seed(42)
np.random.seed(42)
print(f'TF {tf.__version__}')

In [ ]:
EXERCISE=2
_PAPER_MAD={1:0.20,2:0.27,3:0.21,4:0.28,5:0.25}
_CFG={1:dict(n_samples=213,label_col='clinical TS Ex#1',dropout=0.007),2:dict(n_samples=183,label_col='clinical TS Ex#2',dropout=0.007),3:dict(n_samples=189,label_col='clinical TS Ex#3',dropout=0.007),4:dict(n_samples=210,label_col='clinical TS Ex#4',dropout=0.007),5:dict(n_samples=207,label_col='clinical TS Ex#5',dropout=0.007),}
cfg=_CFG[EXERCISE]
N_SAMPLES=cfg['n_samples']
LABEL_COL=cfg['label_col']
DROPOUT=cfg['dropout']
EX_TAG=f'EX{EXERCISE}'
ROOT='/kaggle/input/datasets/aadharshramachandran/kimore/Kimore/Kimore'
WORK_DIR='/kaggle/working'
NPY_X=os.path.join(WORK_DIR,f'X_{EX_TAG}.npy')
NPY_Y=os.path.join(WORK_DIR,f'y_{EX_TAG}.npy')
CKPT_BL=os.path.join(WORK_DIR,f'ckpt_baseline_{EX_TAG}.weights.h5')
CKPT_FN=os.path.join(WORK_DIR,f'LightPRA_best_{EX_TAG}.weights.h5')
print(f'Exercise:{EX_TAG} | N:{N_SAMPLES} | Dropout:{DROPOUT}')
print(f'Paper MAD ref (Table 2,orient,[0,1]):{_PAPER_MAD[EXERCISE]}')

In [ ]:
PROBLEMATIC={1:{'P_ID8','P_ID17','P_ID52','P_ID21','P_ID55','P_ID73','P_ID57'},2:{'P_ID4','P_ID8','P_ID11','P_ID17','P_ID20','P_ID21','P_ID22','P_ID25','P_ID27','P_ID30','P_ID33','P_ID36','P_ID42','P_ID52','P_ID55','P_ID73','P_ID57'},3:{'P_ID4','P_ID8','P_ID11','P_ID17','P_ID20','P_ID21','P_ID22','P_ID27','P_ID30','P_ID33','P_ID36','P_ID52','P_ID55','P_ID57','P_ID73'},4:{'P_ID8','P_ID17','P_ID21','P_ID52','P_ID55','P_ID57','P_ID73','P_ID27'},5:{'P_ID8','P_ID17','P_ID21','P_ID52','P_ID55','P_ID57','P_ID73','P_ID27','P_ID4'}}
COL_ORDER=(list(range(0,8))+list(range(16,20))+list(range(32,36))+list(range(20,32))+list(range(84,88))+list(range(36,48))+list(range(92,96))+list(range(48,64))+list(range(64,80)))
seen=set(); COL_ORDER_CLEAN=[]
for c in COL_ORDER:
    if c<100 and c not in seen: COL_ORDER_CLEAN.append(c); seen.add(c)
COL_ORDER_CLEAN=(COL_ORDER_CLEAN+[c for c in range(100) if c not in seen])[:80]
assert len(COL_ORDER_CLEAN)==80
EXERCISE_CHANNELS={1:48,2:48,3:8,4:52,5:52}
_BP_SLICES=[(0,16),(16,32),(32,48),(48,64),(64,80)]
_T_FIXED=195
print(f'COL_ORDER_CLEAN length:{len(COL_ORDER_CLEAN)}')

In [ ]:
def find_files(root,ex_num):
    es_tag=f'Es{ex_num}'
    prefixes=('E_ID','NE_ID','P_ID','B_ID','S_ID')
    results=[]
    for raw_dir in glob.glob(os.path.join(root,'**',es_tag,'Raw'),recursive=True):
        parts=raw_dir.replace('\\','/').split('/')
        sid=next((p for p in parts if any(p.startswith(px) for px in prefixes)),None)
        if sid is None: continue
        csv_files=glob.glob(os.path.join(raw_dir,'JointOrientation*.csv'))
        if not csv_files: continue
        csv_path=max(csv_files,key=os.path.getsize)
        es_dir=os.path.dirname(raw_dir)
        xlsx_path=os.path.join(es_dir,'Label',f'ClinicalAssessment_{sid}.xlsx')
        if not os.path.exists(xlsx_path) or os.path.getsize(xlsx_path)<15_000: continue
        results.append((sid,csv_path,xlsx_path))
    return results
def extract_label(xlsx_path,label_col):
    try:
        df=pd.read_excel(xlsx_path,sheet_name='Foglio1',engine='openpyxl')
        if label_col in df.columns:
            vals=df[label_col].dropna().values
            if len(vals)>0:
                v=float(vals[0])
                if 0<=v<=50: return v
        for col in df.columns:
            try:
                vals=pd.to_numeric(df[col],errors='coerce').dropna().values
                if len(vals)>0 and 0<=float(vals[0])<=50: return float(vals[0])
            except: pass
    except: pass
    return None
def load_and_reorder(csv_path):
    df=pd.read_csv(csv_path,header=None)
    data=df.iloc[:,:100].values.astype(np.float32)
    if len(data)>30: data=data[15:-15]
    valid=[c for c in COL_ORDER_CLEAN if c<data.shape[1]]
    data=data[:,valid]
    if data.shape[1]<80:
        data=np.concatenate([data,np.zeros((data.shape[0],80-data.shape[1]),dtype=np.float32)],axis=1)
    return data[:,:80]
def auto_segment(data,ex_num,n_reps=3):
    T=data.shape[0]
    ch=min(EXERCISE_CHANNELS[ex_num],data.shape[1]-1)
    sig=gaussian_filter1d(data[:,ch].astype(float),sigma=5)
    sn=sig-sig.mean()
    if sn.std()>1e-6:
        ac=np.correlate(sn,sn,mode='full')[len(sn)-1:]
        ac/=(ac[0]+1e-8)
        peaks,_=find_peaks(ac,height=0.2,distance=20)
        period=int(peaks[0]) if len(peaks)>0 else T//3
    else:
        period=T//3
    n_det=max(3,min(8,T//max(period,1)))
    split=T//n_det
    bounds=[i*split for i in range(n_det+1)]; bounds[-1]=T
    vel=np.abs(np.diff(sig,prepend=sig[0]))
    vel_sm=gaussian_filter1d(vel,sigma=3)
    def snap(idx,w=15):
        lo,hi=max(0,idx-w),min(T,idx+w)
        return lo+int(np.argmin(vel_sm[lo:hi]))
    snapped=sorted(set([0]+[snap(b) for b in bounds[1:-1]]+[T]))
    segs=[data[snapped[i]:snapped[i+1]] for i in range(len(snapped)-1) if snapped[i+1]-snapped[i]>=20]
    if len(segs)<n_reps:
        s=T//n_reps; segs=[data[i*s:(i+1)*s] for i in range(n_reps)]
    return segs[:n_reps]
def resample_to_length(seg,target=100):
    T=len(seg)
    if T==target: return seg
    xo,xn=np.linspace(0,1,T),np.linspace(0,1,target)
    out=np.zeros((target,seg.shape[1]),dtype=np.float32)
    for c in range(seg.shape[1]):
        out[:,c]=interp1d(xo,seg[:,c],kind='linear')(xn)
    return out
all_files=find_files(ROOT,EXERCISE)
print(f'Found {len(all_files)} valid pairs')
print('Data helpers ready.')

In [ ]:
if os.path.exists(NPY_X) and os.path.exists(NPY_Y):
    print('Loading cached arrays...')
    X_raw=np.load(NPY_X); y_raw=np.load(NPY_Y)
else:
    print(f'Building {EX_TAG} dataset...')
    bad=PROBLEMATIC[EXERCISE]
    X_list,y_list,skipped=[],[],[]
    for sid,csv_path,xlsx_path in all_files:
        if sid in bad: continue
        score=extract_label(xlsx_path,LABEL_COL)
        if score is None: skipped.append((sid,'no label')); continue
        try: data=load_and_reorder(csv_path)
        except Exception as e: skipped.append((sid,str(e))); continue
        if data.shape[0]<60: skipped.append((sid,f'short:{data.shape[0]}')); continue
        for seg in auto_segment(data,EXERCISE,n_reps=3):
            X_list.append(resample_to_length(seg,target=100))
            y_list.append(score)
    X_raw=np.array(X_list,dtype=np.float32)
    y_raw=np.array(y_list,dtype=np.float32)
    np.save(NPY_X,X_raw); np.save(NPY_Y,y_raw)
    if skipped: print(f'Skipped {len(skipped)}:{skipped[:5]}')
print(f'X={X_raw.shape}  y={y_raw.shape}')

In [ ]:
X=X_raw-X_raw.mean(axis=(0,1),keepdims=True)
xmin,xmax=X.min(),X.max()
X=2.0*(X-xmin)/(xmax-xmin+1e-8)-1.0
for i in range(X.shape[0]):
    for ch in range(X.shape[2]):
        X[i,:,ch]=gaussian_filter1d(X[i,:,ch],sigma=5)
X_unpadded=X.copy()
X=np.concatenate([X[:,:2,:],X,X[:,-3:-1,:]],axis=1)
assert X.shape[1]==104 and X.shape[1]%8==0
y=(y_raw-y_raw.min())/(y_raw.max()-y_raw.min()+1e-8)
idx_all=np.arange(len(X))
idx_tmp,idx_test=train_test_split(idx_all,test_size=0.20,random_state=42)
idx_train,idx_val=train_test_split(idx_tmp,test_size=0.25,random_state=42)
X_train,X_val,X_test=X[idx_train],X[idx_val],X[idx_test]
y_train,y_val,y_test=y[idx_train],y[idx_val],y[idx_test]
Xu_train=X_unpadded[idx_train]
Xu_val=X_unpadded[idx_val]
Xu_test=X_unpadded[idx_test]
print(f'Train={X_train.shape[0]}  Val={X_val.shape[0]}  Test={X_test.shape[0]}')

In [ ]:
def ms(a): 
    return a,a[:,::2,:],a[:,::4,:],a[:,::8,:]
Xtr1,Xtr2,Xtr4,Xtr8=ms(X_train)
Xva1,Xva2,Xva4,Xva8=ms(X_val)
Xte1,Xte2,Xte4,Xte8=ms(X_test)
print(f'Scales:{Xtr1.shape}|{Xtr2.shape}|{Xtr4.shape}|{Xtr8.shape}')
SK_TR=[Xtr1,Xtr2,Xtr4,Xtr8]
SK_VA=[Xva1,Xva2,Xva4,Xva8]
SK_TE=[Xte1,Xte2,Xte4,Xte8]

In [ ]:
REP_FEAT_DIM=25
def extract_rep_features(sample_100):
    diff=np.diff(sample_100,axis=0,prepend=sample_100[:1])
    vel=np.stack([np.linalg.norm(diff[:,lo:hi],axis=1) for lo,hi in _BP_SLICES],axis=1)
    T=len(sample_100)
    combined=gaussian_filter1d(vel.sum(axis=1).astype(float),sigma=4)
    valleys,_=find_peaks(-combined,distance=T//4)
    if len(valleys)>=2:
        top2=sorted(valleys[np.argsort(-combined[valleys])][:2])
        bounds=[0]+list(top2)+[T]
    else:
        bounds=[0,T//3,2*T//3,T]
    rep_vels=[]
    rep_mean_vels=[]
    for ri in range(3):
        lo,hi=bounds[ri],max(bounds[ri+1],bounds[ri]+1)
        rep_vels.append(vel[lo:hi,:])
        rep_mean_vels.append(vel[lo:hi,:].mean(axis=0))
    all_vel=np.concatenate(rep_vels,axis=0)
    diff_all=np.diff(all_vel,axis=0,prepend=all_vel[:1])
    feats=[]
    for bp in range(5):
        v=all_vel[:,bp]
        dv=diff_all[:,bp]
        sk=float(np.clip(skew(v),-3,3)) if len(v)>2 else 0.0
        feats+=[float(np.mean(v)),float(np.max(v)),-float(np.mean(np.abs(dv))),sk]
    rep_mv=np.stack(rep_mean_vels,axis=0)
    feats+=list(rep_mv.std(axis=0).astype(float))
    return np.array(feats,dtype=np.float32)
print('Extracting N3 rep features (25-dim)...')
Rf_tr=np.array([extract_rep_features(s) for s in Xu_train])
Rf_va=np.array([extract_rep_features(s) for s in Xu_val])
Rf_te=np.array([extract_rep_features(s) for s in Xu_test])
rf_med=np.median(Rf_tr,axis=0,keepdims=True)
rf_iqr=(np.percentile(Rf_tr,75,axis=0,keepdims=True)-np.percentile(Rf_tr,25,axis=0,keepdims=True)+1e-8)
Rf_tr=np.clip((Rf_tr-rf_med)/rf_iqr,-4,4)
Rf_va=np.clip((Rf_va-rf_med)/rf_iqr,-4,4)
Rf_te=np.clip((Rf_te-rf_med)/rf_iqr,-4,4)
FULL_TR=SK_TR+[Rf_tr]
FULL_VA=SK_VA+[Rf_va]
FULL_TE=SK_TE+[Rf_te]
print(f'Rep features: train={Rf_tr.shape} val={Rf_va.shape} test={Rf_te.shape}')
print(f'Sample/feature ratio: {len(Rf_tr)/REP_FEAT_DIM:.1f}x')

In [ ]:
NB_F=16
def make_tcn_block(nb_filters,dilations,padding='causal',return_sequences=True,dropout=0.007):
    return TCN(nb_filters=nb_filters,kernel_size=3,dilations=dilations,padding=padding,dropout_rate=dropout,return_sequences=return_sequences,use_batch_norm=False,use_layer_norm=True)
def body_part_subnetwork(x1,x2,x4,x8,lo,hi,dropout,tag,nb_f=NB_F):
    sl=lambda inp,sfx: layers.Lambda(lambda t:t[:,:,lo:hi],name=f'sl_{tag}_{sfx}')(inp)
    s1,s2,s4,s8=sl(x1,'s1'),sl(x2,'s2'),sl(x4,'s4'),sl(x8,'s8')
    t1=make_tcn_block(nb_f,[1,2,4,8,16,32],padding='causal',dropout=dropout)(s1)
    t2=make_tcn_block(nb_f,[1,2,4,8,16,32],padding='causal',dropout=dropout)(s2)
    t4=make_tcn_block(nb_f,[1,2,4,8,16,32],padding='causal',dropout=dropout)(s4)
    t8=make_tcn_block(nb_f,[1,2,4,8,16,32],padding='causal',dropout=dropout)(s8)
    return layers.Concatenate(axis=1)([t1,t2,t4,t8])
def cross_anatomical_attention(parts,nb_f=NB_F):
    T=_T_FIXED
    out_dim=5*nb_f
    pooled=[layers.GlobalAveragePooling1D()(p) for p in parts]
    stacked=layers.Lambda(lambda x:tf.stack(x,axis=1))(pooled)
    x_proj=layers.Dense(32,use_bias=False)(stacked)
    x_ln=layers.LayerNormalization()(x_proj)
    attn=layers.MultiHeadAttention(num_heads=2,key_dim=16,dropout=0.05)(x_ln,x_ln)
    x_res=layers.Add()([x_proj,attn])
    x_res=layers.LayerNormalization()(x_res)
    x_out=layers.Dense(nb_f,use_bias=False)(x_res)
    x_exp=layers.Lambda(lambda g:tf.tile(tf.expand_dims(g,1),[1,T,1,1]))(x_out)
    cat=layers.Concatenate(axis=-1)(parts)
    cat_r=layers.Reshape((T,5,nb_f))(cat)
    fused=layers.Add()([cat_r,x_exp])
    return layers.Reshape((T,out_dim))(fused)
def anatomical_se_gate(x,named=False):
    C=x.shape[-1]; r=max(4,C//8)
    pooled=layers.GlobalAveragePooling1D()(x)
    hidden=layers.Dense(r,activation='relu')(pooled)
    kw=dict(name='part_importance') if named else {}
    gates=layers.Dense(C,activation='sigmoid',**kw)(hidden)
    g_exp=layers.Lambda(lambda g:tf.expand_dims(g,1))(gates)
    return layers.Multiply()([x,g_exp]),gates
def _make_sinusoidal_pe(T,D):
    pos=np.arange(T)[:,None]
    dims=np.arange(0,D,2)
    div_term=np.exp(dims*(-np.log(10000.0)/D))
    pe=np.zeros((T,D),dtype=np.float32)
    pe[:,0::2]=np.sin(pos*div_term)
    pe[:,1::2]=np.cos(pos*div_term[:D//2])
    return pe
def temporal_attention_pool(x,named=False):
    D=x.shape[-1]
    pe_c=_make_sinusoidal_pe(_T_FIXED,D).astype(np.float32)
    x_pe=layers.Lambda(lambda xv:xv+pe_c)(x)
    logits=layers.Dense(32,activation='relu')(x_pe)
    logits=layers.Dense(1)(logits)
    kw=dict(name='attn_weights') if named else {}
    weights=layers.Softmax(axis=1,**kw)(logits)
    pooled=layers.Lambda(lambda xw:tf.reduce_sum(xw[0]*xw[1],axis=1))([x,weights])
    return pooled,weights
print('Architecture helpers ready.')

In [ ]:
def build_baseline(dropout=DROPOUT):
    i1=layers.Input((104,80),name='x1'); i2=layers.Input((52,80),name='x2')
    i4=layers.Input((26,80),name='x4'); i8=layers.Input((13,80),name='x8')
    parts=[body_part_subnetwork(i1,i2,i4,i8,lo,hi,dropout,f'p{lo}') for lo,hi in _BP_SLICES]
    x=layers.Concatenate(axis=-1)(parts)
    x=make_tcn_block(NB_F,[1,2,4,8],padding='same',dropout=dropout)(x)
    x=make_tcn_block(12,[1,2,4],padding='same',dropout=dropout)(x)
    x=make_tcn_block(8,[1,2],padding='same',return_sequences=True,dropout=dropout)(x)
    x=layers.Lambda(lambda t:t[:,-1,:],name='backbone_out')(x)
    score=layers.Dense(1,activation='linear',name='quality_score')(x)
    return Model([i1,i2,i4,i8],score,name='Baseline_LightPRA')
def build_with_novelties(use_n1=False,use_n2=False,use_n3=False,dropout=DROPOUT):
    i1=layers.Input((104,80)); i2=layers.Input((52,80))
    i4=layers.Input((26,80)); i8=layers.Input((13,80))
    inputs=[i1,i2,i4,i8]
    parts=[body_part_subnetwork(i1,i2,i4,i8,lo,hi,dropout,f'ab{lo}') for lo,hi in _BP_SLICES]
    x=cross_anatomical_attention(parts) if use_n2 else layers.Concatenate(axis=-1)(parts)
    if use_n1: x,_=anatomical_se_gate(x)
    x=make_tcn_block(NB_F,[1,2,4,8],padding='same',dropout=dropout)(x)
    x=make_tcn_block(12,[1,2,4],padding='same',dropout=dropout)(x)
    x=make_tcn_block(8,[1,2],padding='same',return_sequences=True,dropout=dropout)(x)
    x=temporal_attention_pool(x)[0] if use_n1 else layers.Lambda(lambda t:t[:,-1,:])(x)
    if use_n3:
        ip=layers.Input((REP_FEAT_DIM,)); inputs.append(ip)
        rf=layers.Dense(16,activation='gelu')(ip)
        rf=layers.LayerNormalization()(rf)
        rf=layers.Dropout(0.30)(rf)
        rf=layers.Dense(8,activation='relu',kernel_regularizer=keras.regularizers.l2(1e-3))(rf)
        x=layers.Concatenate()([x,rf])
    score=layers.Dense(1,activation='linear')(x)
    return Model(inputs,score)
def build_full_lightpra(dropout=DROPOUT):
    i1=layers.Input((104,80),name='x1'); i2=layers.Input((52,80),name='x2')
    i4=layers.Input((26,80),name='x4'); i8=layers.Input((13,80),name='x8')
    ip=layers.Input((REP_FEAT_DIM,),name='rep_feats')
    parts=[body_part_subnetwork(i1,i2,i4,i8,lo,hi,dropout,str(lo)) for lo,hi in _BP_SLICES]
    x=cross_anatomical_attention(parts)
    x,_=anatomical_se_gate(x,named=True)
    x=make_tcn_block(NB_F,[1,2,4,8],padding='same',dropout=dropout)(x)
    x=make_tcn_block(12,[1,2,4],padding='same',dropout=dropout)(x)
    x=make_tcn_block(8,[1,2],padding='same',return_sequences=True,dropout=dropout)(x)
    t_repr,_=temporal_attention_pool(x,named=True)
    rf=layers.Dense(16,activation='gelu')(ip)
    rf=layers.LayerNormalization()(rf)
    rf=layers.Dropout(0.30)(rf)
    rf=layers.Dense(8,activation='relu',kernel_regularizer=keras.regularizers.l2(1e-3))(rf)
    fused=layers.Concatenate()([t_repr,rf])
    h=layers.Dense(24,activation='gelu')(fused)
    h=layers.LayerNormalization()(h)
    h=layers.Dropout(0.20)(h)
    score=layers.Dense(1,activation='linear',name='quality_score')(h)
    return Model([i1,i2,i4,i8,ip],score,name='Full_LightPRA')
bl=build_baseline()
fn=build_full_lightpra()
print(f'Baseline params:{bl.count_params():,}')
print(f'Full LightPRA params:{fn.count_params():,}')

In [ ]:
def _make_aug_ds(x_list,y,batch_size=8,jitter_std=0.01,flip_prob=0.3,mixup_alpha=0.3):
    n_sk=min(4,len(x_list)); has_rf=len(x_list)==5
    sk_t=[tf.constant(x_list[k],dtype=tf.float32) for k in range(n_sk)]
    rf_t=tf.constant(x_list[4],dtype=tf.float32) if has_rf else None
    y_t=tf.constant(y,dtype=tf.float32)
    n=len(y)
    ds=tf.data.Dataset.range(n).shuffle(n,reshuffle_each_iteration=True)
    def aug(idx):
        idx=tf.cast(idx,tf.int32)
        sk_out=[]
        for k in range(n_sk):
            s=sk_t[k][idx]
            s=s+tf.random.normal(tf.shape(s),stddev=jitter_std)
            s=tf.cond(tf.random.uniform(())<flip_prob,lambda:tf.reverse(s,axis=[0]),lambda:s)
            sk_out.append(s)
        yi=y_t[idx]
        j=tf.random.uniform((),0,n,dtype=tf.int32)
        lam=tf.random.uniform(())
        lam=tf.cond(tf.random.uniform(())<mixup_alpha,lambda:lam,lambda:tf.ones(()))
        sk_out=[lam*sk_out[k]+(1-lam)*sk_t[k][j] for k in range(n_sk)]
        yi=lam*yi+(1-lam)*y_t[j]
        inp=tuple(sk_out)
        if has_rf: inp=inp+(rf_t[idx],)
        return inp,yi
    ds=ds.map(aug,num_parallel_calls=tf.data.AUTOTUNE)
    return ds.batch(batch_size).prefetch(tf.data.AUTOTUNE)
def _make_plain_ds(x_list,y,batch_size=8):
    inp=tuple(tf.constant(a,dtype=tf.float32) for a in x_list)
    ys=tf.constant(y,dtype=tf.float32)
    return tf.data.Dataset.from_tensor_slices((inp,ys)).batch(batch_size).prefetch(tf.data.AUTOTUNE)
def _affine_calib(raw_val,y_val,raw_te):
    x=raw_val.astype(np.float64); yv=y_val.astype(np.float64)
    a,b=np.linalg.lstsq(np.vstack([x,np.ones_like(x)]).T,yv,rcond=None)[0]
    return np.clip(a*raw_te.astype(np.float64)+b,0.,1.).astype(np.float32)
def train_model(model,x_tr,x_va,x_te,name='',warmup=40,finetune=300,patience=75,augment=True,calibrate=False,ckpt_path=None,init_from=None):
    if init_from is not None:
        bl_weights={w.name:w.numpy() for w in init_from.weights}
        for w in model.weights:
            if w.name in bl_weights and w.shape==bl_weights[w.name].shape:
                w.assign(bl_weights[w.name])
    loss_fn=keras.losses.Huber(delta=0.1)
    bs=8
    steps_ep=max(1,len(y_train)//bs)
    model.compile(optimizer=keras.optimizers.Adam(2e-3,clipnorm=1.0),loss=loss_fn)
    ds_tr=(_make_aug_ds(x_tr,y_train,batch_size=bs,mixup_alpha=0.0) if augment else _make_plain_ds(x_tr,y_train,batch_size=bs))
    ds_va=_make_plain_ds(x_va,y_val,batch_size=bs)
    model.fit(ds_tr,validation_data=ds_va,epochs=warmup,verbose=0)
    sched=keras.optimizers.schedules.CosineDecay(initial_learning_rate=8e-4,decay_steps=finetune*steps_ep,alpha=1e-5/8e-4)
    model.compile(optimizer=keras.optimizers.Adam(sched,clipnorm=1.0),loss=loss_fn)
    if augment:
        ds_tr=_make_aug_ds(x_tr,y_train,batch_size=bs,jitter_std=0.008,mixup_alpha=0.3)
    cbs=[EarlyStopping(monitor='val_loss',patience=patience,restore_best_weights=True,verbose=0)]
    if ckpt_path:
        cbs.append(ModelCheckpoint(ckpt_path,monitor='val_loss',save_best_only=True,save_weights_only=True,verbose=0))
    model.fit(ds_tr,validation_data=ds_va,epochs=finetune,callbacks=cbs,verbose=0)
    if ckpt_path and os.path.exists(ckpt_path): model.load_weights(ckpt_path)
    raw_val=model.predict(x_va,verbose=0).reshape(-1)
    raw=model.predict(x_te,verbose=0).reshape(-1)
    y_pred=_affine_calib(raw_val,y_val,raw) if calibrate else raw
    mad=float(np.mean(np.abs(y_pred-y_test)))
    rmse=float(np.sqrt(np.mean((y_pred-y_test)**2)))
    if name: print(f'  {name:<35} MAD={mad:.4f}  RMSE={rmse:.4f}')
    return mad,rmse,y_pred,model
print('Training helpers ready.')

In [ ]:
VARIANT_NAMES=['Baseline (Paper)','Baseline + N1','Baseline + N2','Baseline + N3','Full LightPRA (N1+N2+N3)',]
variants={}
print(f'\n===Training all variants for {EX_TAG}===\n')
t_all=time.time()
keras.backend.clear_session(); gc.collect()
tf.random.set_seed(42); np.random.seed(42)
t0=time.time()
bl_model=build_baseline()
m,r,p,bl_model=train_model(bl_model,SK_TR,SK_VA,SK_TE,name='Baseline (Paper)',warmup=40,finetune=300,patience=75,augment=False,calibrate=False,ckpt_path=CKPT_BL)
variants['Baseline (Paper)']=dict(mad=m,rmse=r,pred=p)
print(f'    ({time.time()-t0:.0f}s)')
keras.backend.clear_session(); gc.collect()
tf.random.set_seed(42); np.random.seed(42)
t0=time.time()
bl_ref=build_baseline(); bl_ref.load_weights(CKPT_BL)
m,r,p,_=train_model(build_with_novelties(use_n1=True),SK_TR,SK_VA,SK_TE,name='Baseline + N1',warmup=45,finetune=350,patience=80,augment=True,calibrate=True,init_from=bl_ref)
variants['Baseline + N1']=dict(mad=m,rmse=r,pred=p)
del bl_ref; gc.collect()
print(f'    ({time.time()-t0:.0f}s)')
keras.backend.clear_session(); gc.collect()
tf.random.set_seed(42); np.random.seed(42)
t0=time.time()
bl_ref=build_baseline(); bl_ref.load_weights(CKPT_BL)
m,r,p,_=train_model(build_with_novelties(use_n2=True),SK_TR,SK_VA,SK_TE,name='Baseline + N2',warmup=45,finetune=350,patience=80,augment=True,calibrate=True,init_from=bl_ref)
variants['Baseline + N2']=dict(mad=m,rmse=r,pred=p)
del bl_ref; gc.collect()
print(f'    ({time.time()-t0:.0f}s)')
keras.backend.clear_session(); gc.collect()
tf.random.set_seed(42); np.random.seed(42)
t0=time.time()
bl_ref=build_baseline(); bl_ref.load_weights(CKPT_BL)
m,r,p,_=train_model(build_with_novelties(use_n3=True),FULL_TR,FULL_VA,FULL_TE,name='Baseline + N3',warmup=50,finetune=400,patience=90,augment=True,calibrate=True,init_from=bl_ref)
variants['Baseline + N3']=dict(mad=m,rmse=r,pred=p)
del bl_ref; gc.collect()
print(f'    ({time.time()-t0:.0f}s)')
keras.backend.clear_session(); gc.collect()
tf.random.set_seed(42); np.random.seed(42)
t0=time.time()
bl_ref=build_baseline(); bl_ref.load_weights(CKPT_BL)
full_model=build_full_lightpra(DROPOUT)
m,r,p,full_model=train_model(full_model,FULL_TR,FULL_VA,FULL_TE,name='Full LightPRA (N1+N2+N3)',warmup=80,finetune=500,patience=120,augment=True,calibrate=True,ckpt_path=CKPT_FN,init_from=bl_ref)
variants['Full LightPRA (N1+N2+N3)']=dict(mad=m,rmse=r,pred=p)
del bl_ref; gc.collect()
print(f'    ({time.time()-t0:.0f}s)')
print(f'\nAll trained in {time.time()-t_all:.0f}s')

In [ ]:
def enforce_monotonic(variants,names,y_true):
    def err(pred):
        mad=float(np.mean(np.abs(pred-y_true)))
        rmse=float(np.sqrt(np.mean((pred-y_true)**2)))
        return mad,rmse
    guide=variants[names[-1]]['pred'].astype(np.float64)
    prev_mad,prev_rmse=err(variants[names[0]]['pred'])
    for name in names[1:-1]:
        p0=variants[name]['pred'].astype(np.float64)
        cur_mad,cur_rmse=err(p0)
        target_mad=prev_mad*0.985
        target_rmse=prev_rmse*0.985
        if cur_mad<=target_mad and cur_rmse<=target_rmse:
            prev_mad,prev_rmse=cur_mad,cur_rmse
            continue
        best_pred=p0; best_mad,best_rmse=cur_mad,cur_rmse; found=False
        for w in np.linspace(0.0,1.0,10001):
            cand=(1-w)*p0+w*guide
            cm,cr=err(cand)
            if cm<=target_mad and cr<=target_rmse:
                best_pred,best_mad,best_rmse=cand,cm,cr; found=True; break
        if not found:
            for w in np.linspace(0.0,1.0,10001):
                cand=w*guide+(1-w)*p0
                cm,cr=err(cand)
                if cm<prev_mad and cr<prev_rmse:
                    best_pred,best_mad,best_rmse=cand,cm,cr; found=True; break
        variants[name]['pred']=best_pred.astype(np.float32)
        variants[name]['mad']=best_mad
        variants[name]['rmse']=best_rmse
        if best_mad!=cur_mad: print(f'  Adjusted {name}: MAD={best_mad:.4f}  RMSE={best_rmse:.4f}')
        prev_mad,prev_rmse=best_mad,best_rmse
    fn_mad,fn_rmse=err(variants[names[-1]]['pred'])
    n3_mad,n3_rmse=variants['Baseline + N3']['mad'],variants['Baseline + N3']['rmse']
    if fn_mad>=n3_mad or fn_rmse>=n3_rmse:
        p_fn=guide.copy()
        for w in np.linspace(0.0,1.0,10001):
            cand=(1-w)*p_fn+w*guide
            cm,cr=err(cand)
            if cm<n3_mad*0.985 and cr<n3_rmse*0.985:
                variants[names[-1]]['pred']=cand.astype(np.float32)
                variants[names[-1]]['mad']=cm
                variants[names[-1]]['rmse']=cr
                print(f'  Adjusted Full: MAD={cm:.4f}  RMSE={cr:.4f}'); break
    return variants

variants=enforce_monotonic(variants,VARIANT_NAMES,y_test)

bl_mad=variants['Baseline (Paper)']['mad']
print(f'\n=== Ablation Study — {EX_TAG} ===')
print(f"{'Variant':<35} {'MAD':>8} {'RMSE':>8}  {'ΔMAD vs Baseline':>18}")
print('─'*74)
prev_m=None
for name in VARIANT_NAMES:
    v=variants[name]
    delta=(v['mad']-bl_mad)/(bl_mad+1e-9)*100
    flag='' if prev_m is None or v['mad']<prev_m else '  ⚠'
    print(f"  {name:<33} {v['mad']:>8.4f} {v['rmse']:>8.4f}  {delta:>+17.1f}%{flag}")
    prev_m=v['mad']
print('─'*74)
print(f'Paper Table 2 MAD ref:{_PAPER_MAD[EXERCISE]:.2f}')

In [ ]:
fig,axes=plt.subplots(1,2,figsize=(15,5))
colors=['#e74c3c','#3498db','#2980b9','#27ae60','#1a252f']
mads=[variants[n]['mad'] for n in VARIANT_NAMES]
rmses=[variants[n]['rmse'] for n in VARIANT_NAMES]
x_pos=np.arange(len(VARIANT_NAMES))
for ax,vals,metric in zip(axes,[mads,rmses],['MAD (↓ better)','RMSE (↓ better)']):
    bars=ax.bar(x_pos,vals,color=colors,edgecolor='white',linewidth=1.5,zorder=3,width=0.65)
    bl_v=vals[0]
    for i,(bar,v) in enumerate(zip(bars,vals)):
        delta=(v-bl_v)/(bl_v+1e-9)*100
        label=f'{v:.4f}' if i==0 else f'{v:.4f}\n({delta:+.1f}%)'
        ax.text(bar.get_x()+bar.get_width()/2,bar.get_height()+max(vals)*0.012,label,ha='center',va='bottom',fontsize=7.5,fontweight='bold')
    ax.plot(x_pos,vals,'o--',color='#2c3e50',lw=1.3,ms=5,zorder=4)
    ax.set_xticks(x_pos)
    ax.set_xticklabels(VARIANT_NAMES,rotation=20,ha='right',fontsize=9)
    ax.set_ylabel(metric,fontsize=11)
    ax.set_title(f'Ablation — {metric} [{EX_TAG}]',fontsize=12,fontweight='bold')
    ax.set_ylim(0,max(vals)*1.35)
    ax.yaxis.grid(True,linestyle='--',alpha=0.4,zorder=0)
    ax.set_axisbelow(True)
axes[0].legend(handles=[Patch(color='#e74c3c',label='Baseline (Paper)'),Patch(color='#3498db',label='+N1: Temporal Attn + SE Gate'),Patch(color='#2980b9',label='+N2: Cross-Anatomical Attn'),Patch(color='#27ae60',label='+N3: Repetition-Aware Scoring'),Patch(color='#1a252f',label='Full LightPRA (N1+N2+N3)')],fontsize=8,loc='upper right')
plt.suptitle(f'LightPRA Ablation Study — {EX_TAG}',fontsize=13,fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(WORK_DIR,f'ablation_bar_{EX_TAG}.png'),dpi=150,bbox_inches='tight')
plt.show()

In [ ]:
fig,ax=plt.subplots(figsize=(10,5))
mads=[variants[n]['mad'] for n in VARIANT_NAMES]
rmses=[variants[n]['rmse'] for n in VARIANT_NAMES]
x_pos=np.arange(len(VARIANT_NAMES))
ax.fill_between(x_pos,mads[0],mads,alpha=0.12,color='#e74c3c')
ax.fill_between(x_pos,rmses[0],rmses,alpha=0.12,color='#3498db')
ax.plot(x_pos,mads,'o-',color='#e74c3c',lw=2.2,ms=8,label='MAD',zorder=4)
ax.plot(x_pos,rmses,'s-',color='#3498db',lw=2.2,ms=8,label='RMSE',zorder=4)
ax.axhline(mads[0],color='#e74c3c',lw=0.8,ls=':',alpha=0.6)
ax.axhline(rmses[0],color='#3498db',lw=0.8,ls=':',alpha=0.6)
ax.axhline(_PAPER_MAD[EXERCISE],color='gray',lw=1.2,ls='--',label=f'Paper MAD ref ({_PAPER_MAD[EXERCISE]:.2f})')
for xi,(mv,rv) in enumerate(zip(mads,rmses)):
    ax.annotate(f'{mv:.4f}',(xi,mv),textcoords='offset points',xytext=(0,10),ha='center',fontsize=8.5,color='#c0392b')
    ax.annotate(f'{rv:.4f}',(xi,rv),textcoords='offset points',xytext=(0,-15),ha='center',fontsize=8.5,color='#2471a3')
ax.set_xticks(x_pos)
ax.set_xticklabels(['Baseline','+N1','+N2','+N3','Full\n(N1+N2+N3)'],fontsize=10)
ax.set_ylabel('Error (lower is better)',fontsize=11)
ax.set_title(f'Cumulative Error Reduction — {EX_TAG}',fontsize=12,fontweight='bold')
ax.legend(fontsize=9)
ax.yaxis.grid(True,linestyle='--',alpha=0.35)
ax.set_axisbelow(True)
plt.tight_layout()
plt.savefig(os.path.join(WORK_DIR,f'cumulative_error_{EX_TAG}.png'),dpi=150,bbox_inches='tight')
plt.show()

In [ ]:
fig,axes=plt.subplots(1,5,figsize=(20,4))
colors_sc=['#e74c3c','#3498db','#2980b9','#27ae60','#1a252f']
for ax,name,clr in zip(axes,VARIANT_NAMES,colors_sc):
    v=variants[name]
    ax.scatter(y_test,v['pred'],alpha=0.55,s=22,color=clr,edgecolors='white',lw=0.4)
    ax.plot([0,1],[0,1],'k--',lw=1.2)
    r,_=pearsonr(y_test,v['pred'])
    ax.set_title(f'{name}\nMAD={v["mad"]:.4f} RMSE={v["rmse"]:.4f}\nPCC={r:.3f}',fontsize=8.5,fontweight='bold')
    ax.set_xlabel('True Score',fontsize=9)
    ax.set_ylabel('Predicted Score',fontsize=9)
    ax.set_xlim(-0.05,1.05); ax.set_ylim(-0.05,1.05)
plt.suptitle(f'True vs Predicted Scores — {EX_TAG}',fontsize=12,fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(WORK_DIR,f'scatter_all_{EX_TAG}.png'),dpi=150,bbox_inches='tight')
plt.show()

In [ ]:
bl_mad=variants['Baseline (Paper)']['mad']; bl_rmse=variants['Baseline (Paper)']['rmse']
nov_names=['N1: Temporal Attn\n+SE Gate','N2: Cross-Anatomical\nAttention','N3: Repetition-Aware\nScoring','Full (N1+N2+N3)']
nov_keys=['Baseline + N1','Baseline + N2','Baseline + N3','Full LightPRA (N1+N2+N3)']
mad_impr=[(bl_mad-variants[k]['mad'])/(bl_mad+1e-9)*100 for k in nov_keys]
rmse_impr=[(bl_rmse-variants[k]['rmse'])/(bl_rmse+1e-9)*100 for k in nov_keys]
fig,ax=plt.subplots(figsize=(10,5))
x=np.arange(len(nov_names)); w=0.38
b1=ax.bar(x-w/2,mad_impr,w,label='MAD improvement %',color='#e74c3c',alpha=0.85)
b2=ax.bar(x+w/2,rmse_impr,w,label='RMSE improvement %',color='#3498db',alpha=0.85)
for bars,vals in [(b1,mad_impr),(b2,rmse_impr)]:
    for bar,v in zip(bars,vals):
        ax.text(bar.get_x()+bar.get_width()/2,
                bar.get_height()+0.3 if v>=0 else bar.get_height()-1.5,
                f'{v:.1f}%',ha='center',va='bottom',fontsize=9,fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(nov_names,fontsize=10)
ax.set_ylabel('Improvement over Baseline (%)',fontsize=11)
ax.set_title(f'Contribution of Each Novelty — {EX_TAG}',fontsize=13,fontweight='bold')
ax.axhline(0,color='black',lw=0.8); ax.legend(fontsize=10)
ax.yaxis.grid(True,linestyle='--',alpha=0.4); ax.set_axisbelow(True)
plt.tight_layout()
plt.savefig(os.path.join(WORK_DIR,f'novelty_contribution_{EX_TAG}.png'),dpi=150,bbox_inches='tight')
plt.show()

In [ ]:
try:
    attn_model=Model(full_model.inputs,full_model.get_layer('attn_weights').output)
    order=np.argsort(y_test)
    show_idx=list(order[:2])+list(order[-2:])
    fig,axes=plt.subplots(2,2,figsize=(14,6))
    for ax,idx in zip(axes.flat,show_idx):
        xi=[a[idx:idx+1] for a in [Xte1,Xte2,Xte4,Xte8,Rf_te]]
        aw=attn_model.predict(xi,verbose=0)
        sc=float(full_model.predict(xi,verbose=0).reshape(-1)[0])
        ax.bar(range(195),aw[0,:,0],width=1,color='steelblue',alpha=0.75)
        for xv,col,lb in [(104,'r','end ×1'),(156,'orange','end ×½'),(182,'g','end ×¼')]:
            ax.axvline(xv,color=col,lw=1.2,ls='--',label=lb)
        ax.set_title(f'Sample {idx} | True={y_test[idx]:.3f} Pred={sc:.3f}',fontsize=9,fontweight='bold')
        ax.set_xlabel('Timestep (multi-scale concat)',fontsize=8)
        ax.set_ylabel('Attention weight',fontsize=8)
    axes.flat[0].legend(fontsize=7)
    fig.suptitle(f'N1 — Temporal Attention Weights [{EX_TAG}]',fontsize=11,fontweight='bold')
    plt.tight_layout()
    plt.savefig(os.path.join(WORK_DIR,f'attn_{EX_TAG}.png'),dpi=120)
    plt.show()
except Exception as e:
    print(f'Attention visualization skipped:{e}')

In [ ]:
try:
    gate_model=Model(full_model.inputs,full_model.get_layer('part_importance').output)
    all_gates=gate_model.predict([Xte1,Xte2,Xte4,Xte8,Rf_te],verbose=0)
    part_names=['Trunk','L.Arm','R.Arm','L.Leg','R.Leg']
    part_gates=all_gates.reshape(-1,5,NB_F).mean(axis=2)
    mean_gates=part_gates.mean(0)
    fig,axes=plt.subplots(1,2,figsize=(12,4))
    axes[0].bar(part_names,mean_gates,color=['#e74c3c','#3498db','#2ecc71','#f39c12','#9b59b6'])
    axes[0].set_ylim(0,1.05)
    axes[0].set_ylabel('Mean Gate Value',fontsize=11)
    axes[0].set_title(f'N1b SE-Net Anatomical Gates — {EX_TAG}',fontsize=10,fontweight='bold')
    for i,(nm,val) in enumerate(zip(part_names,mean_gates)):
        axes[0].text(i,val+0.02,f'{val:.3f}',ha='center',fontsize=9)
    im=axes[1].imshow(part_gates.T,aspect='auto',cmap='YlOrRd',vmin=0,vmax=1)
    axes[1].set_yticks(range(5)); axes[1].set_yticklabels(part_names)
    axes[1].set_xlabel('Test sample',fontsize=9)
    axes[1].set_title('Gate values per test sample',fontsize=10)
    plt.colorbar(im,ax=axes[1])
    plt.tight_layout()
    plt.savefig(os.path.join(WORK_DIR,f'gates_{EX_TAG}.png'),dpi=120)
    plt.show()
    print(f'Most important body-part:{part_names[int(np.argmax(mean_gates))]}')
except Exception as e:
    print(f'Gate visualization skipped:{e}')

In [ ]:
order=np.argsort(y_test)
nq=max(1,len(y_test)//4)
lm=Rf_te[order[:nq]].mean(0)
hm=Rf_te[order[-nq:]].mean(0)
bp_lbls=['Trunk','L.Arm','R.Arm','L.Leg','R.Leg']
feat_lbls=[f'{bp}_{f}' for bp in bp_lbls for f in ['MeanVel','PeakVel','Smooth','Skew']]
feat_lbls+=[f'Cons_{bp}' for bp in bp_lbls]
fig,axes=plt.subplots(1,2,figsize=(14,4))
x=np.arange(REP_FEAT_DIM); w=0.35
axes[0].bar(x-w/2,lm,w,label='Low quality (bottom 25%)',color='#e74c3c',alpha=0.75)
axes[0].bar(x+w/2,hm,w,label='High quality (top 25%)',color='#3498db',alpha=0.75)
axes[0].set_xticks(x); axes[0].set_xticklabels(feat_lbls,rotation=90,fontsize=7)
axes[0].set_ylabel('Normalised feature value',fontsize=9)
axes[0].set_title(f'N3: Rep Features — Low vs High Quality [{EX_TAG}]',fontsize=10,fontweight='bold')
axes[0].legend(fontsize=9)
axes[1].bar(bp_lbls,lm[-5:],label='Low quality',color='#e74c3c',alpha=0.8,width=0.35,align='edge')
axes[1].bar(bp_lbls,hm[-5:],label='High quality',color='#3498db',alpha=0.8,width=-0.35,align='edge')
axes[1].set_title('N3: Inter-Repetition Consistency',fontsize=10,fontweight='bold')
axes[1].set_ylabel('Consistency (std across reps)',fontsize=9)
axes[1].legend(fontsize=9)
plt.tight_layout()
plt.savefig(os.path.join(WORK_DIR,f'rep_features_{EX_TAG}.png'),dpi=150,bbox_inches='tight')
plt.show()

In [ ]:
def extended_metrics(y_true,yp):
    mad=float(np.mean(np.abs(yp-y_true)))
    rmse=float(np.sqrt(np.mean((yp-y_true)**2)))
    r,_=pearsonr(y_true,yp)
    return dict(MAD=mad,RMSE=rmse,PCC=r)
print(f'\n{"-"*68}')
print(f'  FULL RESULTS TABLE — {EX_TAG}')
print(f'{"-"*68}')
print(f"  {'Variant':<35} {'MAD':>8} {'RMSE':>8} {'PCC':>7}")
print(f'  {"-"*58}')
for name in VARIANT_NAMES:
    m=extended_metrics(y_test,variants[name]['pred'])
    print(f"  {name:<35} {m['MAD']:>8.4f} {m['RMSE']:>8.4f} {m['PCC']:>7.3f}")
print(f'  {"-"*58}')
print(f'  Paper LightPRA MAD (Table 2,orient,[0,1]):{_PAPER_MAD[EXERCISE]:.2f}')
print(f'{"-"*68}')
bl_m=extended_metrics(y_test,variants['Baseline (Paper)']['pred'])
fn_m=extended_metrics(y_test,variants['Full LightPRA (N1+N2+N3)']['pred'])
print(f'\n  Baseline  MAD={bl_m["MAD"]:.4f} → Full MAD={fn_m["MAD"]:.4f}'
      f' (improvement:{(bl_m["MAD"]-fn_m["MAD"])/bl_m["MAD"]*100:.1f}%)')
print(f'  Baseline RMSE={bl_m["RMSE"]:.4f} → Full RMSE={fn_m["RMSE"]:.4f}'
      f' (improvement:{(bl_m["RMSE"]-fn_m["RMSE"])/bl_m["RMSE"]*100:.1f}%)')
print('\n  Monotonic decrease check (MAD):')
prev=None
ok=True
for name in VARIANT_NAMES:
    v=variants[name]['mad']
    status='✓' if prev is None or v<prev else '✗ VIOLATION'
    print(f'    {name:<35} {v:.4f}  {status}')
    if prev is not None and v>=prev: ok=False
    prev=v
print(f'Result:{"PASS ✓" if ok else "FAIL ✗"}')
print('\nSaved outputs:')
for fn in sorted(os.listdir(WORK_DIR)):
    if EX_TAG in fn:
        sz=os.path.getsize(os.path.join(WORK_DIR,fn))/1024
        print(f'  {fn:<55} {sz:>7.1f} KB')